Project Environment Setup

In [1]:
pip install pandas scikit-learn xgboost joblib streamlit tableau-api-lib

Note: you may need to restart the kernel to use updated packages.


Step 1: Read the Dataset and Create a DataFrame

In [2]:
import pandas as pd

# Read the dataset
try:
    df = pd.read_csv("sarcopenia-data-set_columns_corrected_13_optimized_sampled.csv")
    print("DataFrame successfully created.")
    print("First 5 rows:")
    print(df.head())
    print("\nDataFrame info:")
    df.info()
except FileNotFoundError:
    print("Error: 'sarcopenia-data-set_columns_corrected_13_optimized-sampled.csv' not found. Ensure the file is in the correct directory.")
    exit() # Exit if the file is not found

DataFrame successfully created.
First 5 rows:
     AT  Age_Group_AGE 60-80    BMI   CST  DM_Type2  Exercise_Status_3-4/week  \
0  43.5                    1  38.44   9.6         1                         0   
1  43.3                    0  38.86  10.2         0                         0   
2  31.4                    1  30.90  15.3         1                         0   
3  51.0                    1  34.70  15.1         1                         0   
4  33.8                    0  31.50  10.3         1                         1   

   Gait_Speed  Gender_M  Grip_Str  OP  STAR  Total_Number_of_Chronic_Diseases  \
0        0.76         0      20.0   1  1.13                                 4   
1        0.94         0      25.0   0  1.11                                 1   
2        0.95         0      18.0   0  1.02                                 3   
3        0.74         1      28.0   0  1.47                                 6   
4        1.23         1      40.0   0  1.07                   

Step 2: Separate Features and Target Columns

In [3]:
# Separate 'Sarcopenia' column as the target (y)
# Separate all other columns as features (X)
if 'Sarcopenia' in df.columns:
    X = df.drop('Sarcopenia', axis=1)
    y = df['Sarcopenia']
    print("\nFeatures (X) and Target (y) successfully separated.")
    print("First 5 rows of X:")
    print(X.head())
    print("\nFirst 5 values of y:")
    print(y.head())
else:
    print("Error: 'Sarcopenia' column not found in the DataFrame. Please check and update the target column name.")
    exit() # Exit if the target column is not found


Features (X) and Target (y) successfully separated.
First 5 rows of X:
     AT  Age_Group_AGE 60-80    BMI   CST  DM_Type2  Exercise_Status_3-4/week  \
0  43.5                    1  38.44   9.6         1                         0   
1  43.3                    0  38.86  10.2         0                         0   
2  31.4                    1  30.90  15.3         1                         0   
3  51.0                    1  34.70  15.1         1                         0   
4  33.8                    0  31.50  10.3         1                         1   

   Gait_Speed  Gender_M  Grip_Str  OP  STAR  Total_Number_of_Chronic_Diseases  \
0        0.76         0      20.0   1  1.13                                 4   
1        0.94         0      25.0   0  1.11                                 1   
2        0.95         0      18.0   0  1.02                                 3   
3        0.74         1      28.0   0  1.47                                 6   
4        1.23         1      40.0   

Step 3: Split into Training and Test Data

In [4]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # stratify to maintain class balance

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (3999, 17)
X_test shape: (1000, 17)
y_train shape: (3999,)
y_test shape: (1000,)


Step 4: Create Model with XGBClassifier and Train with GridSearchCV

In [7]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

# Create XGBClassifier model
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)

# Define hyperparameter range for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.001, 0.01] # L1 regularization
}

 # Stratified K-Fold for robust evaluation on the training data
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create GridSearchCV object
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=cv_splitter, scoring='roc_auc', n_jobs=-1, verbose=2)

print("\nStarting model training with GridSearchCV...")
# Train the model
grid_search.fit(X_train, y_train)

print("\nGridSearchCV training completed.")


Starting model training with GridSearchCV...
Fitting 5 folds for each of 2187 candidates, totalling 10935 fits

GridSearchCV training completed.


Step 5: Obtain the Best Model and Assign to model Variable

In [8]:
# Get the best model
model = grid_search.best_estimator_

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
print("\nBest model assigned to 'model' variable.")


Best parameters: {'colsample_bytree': 0.9, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'reg_alpha': 0.01, 'subsample': 0.9}
Best cross-validation score: 0.9997

Best model assigned to 'model' variable.


Step 6: Test the Model

In [9]:
# Make predictions on the test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1_score = f1_score(y_test, y_pred)
roc_auc_score = roc_auc_score(y_test, y_prob)

class_report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.4f}")
print(f"\nModel Precision: {precision:.4f}")
print(f"\nModel Recall: {recall:.4f}")
print(f"\nModel F1-Score: {f1_score:.4f}")
print(f"\nModel Roc_Auc_Score: {roc_auc_score:.4f}")
print("\nClassification Report:")
print(class_report)
print("\nConfusion Matrix:")
print(conf_matrix)


Model Accuracy: 0.9960

Model Precision: 1.0000

Model Recall: 0.9804

Model F1-Score: 0.9901

Model Roc_Auc_Score: 0.9956

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       796
           1       1.00      0.98      0.99       204

    accuracy                           1.00      1000
   macro avg       1.00      0.99      0.99      1000
weighted avg       1.00      1.00      1.00      1000


Confusion Matrix:
[[796   0]
 [  4 200]]


Step 7: Save the Model to Disk

In [10]:
from xgboost import XGBClassifier
import joblib
import os

# Create the directory to save the model
model_dir = "model"
os.makedirs(model_dir, exist_ok=True)

# Save the model
json_path = os.path.join(model_dir, "xgb_model.json")
booster = model.get_booster()
booster.save_model(json_path)

print(f"Model .json olarak kaydedildi: {json_path}")

Model .json olarak kaydedildi: model\xgb_model.json


Step 8: Ensure Columns in Inference File Match the Original DataFrame

In [11]:
# Save column names to a file (can be read later in inference.py)
import json

columns_file_path = os.path.join(model_dir, "feature_columns.json")
with open(columns_file_path, 'w') as f:
    json.dump(X.columns.tolist(), f)

print(f"\nFeature column names saved to '{columns_file_path}'.")

# We will read these column names within input_fn.


Feature column names saved to 'model\feature_columns.json'.


Step 9: Create inference.py and requirements.txt Files and Upload to the Model's Saved Folder

In [12]:
%%writefile model/inference.py

import os
import json
import pandas as pd
import xgboost as xgb
from io import StringIO

# Global variable to store column names
FEATURE_COLUMNS = None

def model_fn(model_dir):
    """
    Loads the model. This function is called to load the model in the deployment environment.
    """
    global FEATURE_COLUMNS
    model = xgb.Booster()
    model.load_model(os.path.join(model_dir, "xgb_model.json"))
    
    # Load the file containing column names
    columns_file_path = os.path.join(model_dir, "feature_columns.json")
    with open(columns_file_path, 'r') as f:
        FEATURE_COLUMNS = json.load(f)
    
    return model

def input_fn(request_body, request_content_type):
    """
    Converts the incoming request into a format the model can make predictions on.
    """
    if FEATURE_COLUMNS is None:
        raise RuntimeError("FEATURE_COLUMNS not loaded yet. model_fn must have been run.")

    if request_content_type == "text/csv":
        data = StringIO(request_body)
        df = pd.read_csv(data)
    elif request_content_type == "application/json":
        data = json.loads(request_body)
        # Convert to DataFrame when single row data comes from JSON
        if isinstance(data, dict):
            df = pd.DataFrame([data])
        # Convert to DataFrame when multiple rows data comes from JSON
        elif isinstance(data, list):
            df = pd.DataFrame(data)
        else:
            raise ValueError("Unsupported JSON format")
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")

    # Reorder the columns of the incoming DataFrame according to the expected column order
    # Fill missing columns with 0 or an appropriate default value
    for col in FEATURE_COLUMNS:
        if col not in df.columns:
            df[col] = 0.0
    
    # Sort columns by expected order
    df = df[FEATURE_COLUMNS]

    return df

def predict_fn(input_data, model):
    """
    Makes predictions using the model.
    """
    dmatrix = xgb.DMatrix(input_data)
    predictions = model.predict(dmatrix)
    # Optional: convert float probabilities to binary class labels
    predictions = [int(round(p)) for p in predictions]
    return predictions

def output_fn(prediction, accept_content_type):
    """
    Converts the model's prediction into the desired output format.
    """
    if accept_content_type == "application/json":
        return json.dumps(prediction), accept_content_type
    elif accept_content_type == "text/csv":
        return ",".join(map(str, prediction)), accept_content_type
    else:
        raise ValueError(f"Unsupported output type: {accept_content_type}")


Writing model/inference.py


In [13]:
%%writefile model/requirements.txt
pandas
scikit-learn
xgboost
joblib
json
os
io


Writing model/requirements.txt


Step 11: Create a Model Object for Deployment

In [1]:
# We are preparing a Git repository structure for Hugging Face Spaces
# The model and inference files are ready in the model/ directory.
# Now we will prepare the Streamlit application.

In [19]:
X.dtypes.tolist()

[dtype('float64'),
 dtype('int64'),
 dtype('float64'),
 dtype('float64'),
 dtype('int64'),
 dtype('int64'),
 dtype('float64'),
 dtype('int64'),
 dtype('float64'),
 dtype('int64'),
 dtype('float64'),
 dtype('int64'),
 dtype('float64'),
 dtype('float64'),
 dtype('int64'),
 dtype('float64'),
 dtype('float64')]

In [ ]:
X.dtypes.tolist()

In [27]:
data_types_list = []
for col in X.columns:
    data_types_list.append({'Column Name': col, 'Data Type': str(X[col].dtype)})

data_types_df = pd.DataFrame(data_types_list)

output_file_name = 'data_types.csv'
data_types_df.to_csv(output_file_name, index=False, encoding='utf-8')

print(f"Data Types ' is saved to {output_file_name}'.")

Data Types ' is saved to data_types.csv'.


Creating Dashboards with Tableau

In [28]:
# Create a DataFrame containing predictions
y_pred_df = pd.DataFrame(y_pred, columns=['Predicted_Sarcopenia_Risk'], index=X_test.index)
test_results_df = pd.concat([X_test, y_test.rename('Actual_Sarcopenia_Risk'), y_pred_df], axis=1)

# Save this DataFrame to a CSV file
tableau_data_path = "sarcopenia_prediction_results_for_tableau.csv"
test_results_df.to_csv(tableau_data_path, index=False)
print(f"\nData for Tableau saved to '{tableau_data_path}'.")


Data for Tableau saved to 'sarcopenia_prediction_results_for_tableau.csv'.
